# Notebook principal
Este notebook contém o fluxo de pré-processamento, treino e explicações SHAP. Execute as células abaixo em ordem.

# Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report
import ipaddress
import shap
from lime.lime_tabular import LimeTabularExplainer

c:\Users\vinic\.virtualenvs\RansonwareData-Oy5xU5Qp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Lendo o dataset

In [3]:
df = pd.read_csv('../Datasets/LMD-2023-dataset.csv')

C:\Users\vinic\AppData\Local\Temp\ipykernel_20144\2502173159.py:1: DtypeWarning: Columns (40,53,56,57,58,59,60,61,62,65,66,68,69,70,71,72,73,74,75,76,77,78,79,81,82,83,84,85,87,88,89,90,91,92,93) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Datasets/LMD-2023-dataset.csv')


# Começando o treinamento

In [4]:
drop_cols = [
    'Guid', 'EventRecordID', 'ProcessGuid', 'ParentProcessGuid', 
    'LogonGuid', 'Hashes', 'ID', 'SourceProcessGUID', 'TargetProcessGUID',
    'CreationUtcTime', 'PreviousCreationUtcTime',
    'CommandLine', 'ParentCommandLine', 'OriginalFileName', 'Description'
]

y = df['Label']
X = df.drop(columns=['Label'] + drop_cols)

In [5]:
for col in ['SystemTime', 'UtcTime']:
    X[col] = pd.to_datetime(X[col], errors='coerce')
    X[col + '_hour'] = X[col].dt.hour.fillna(0)
    X[col + '_weekday'] = X[col].dt.weekday.fillna(0)
X = X.drop(columns=['SystemTime', 'UtcTime'])

# ---------- 3️⃣ Processar IPs ----------
def ip_to_int(ip):
    try:
        return int(ipaddress.ip_address(ip))
    except:
        return 0

for col in ['SourceIp', 'DestinationIp']:
    X[col] = X[col].astype(str).apply(ip_to_int)

# ---------- 4️⃣ Separar colunas numéricas e categóricas ----------
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# ---------- 5️⃣ Label Encoding ----------
X_encoded = X.copy()
for col in categorical_features:
    X_encoded[col] = LabelEncoder().fit_transform(X_encoded[col].astype(str))


C:\Users\vinic\AppData\Local\Temp\ipykernel_20144\3538020010.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X[col] = pd.to_datetime(X[col], errors='coerce')


In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

rf.fit(X_train, y_train)


RandomForestClassifier(n_jobs=-1, random_state=42)

# Testes da IA

In [7]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    322310
           1       1.00      1.00      1.00     22202
           2       1.00      1.00      1.00      6056

    accuracy                           1.00    350568
   macro avg       1.00      1.00      1.00    350568
weighted avg       1.00      1.00      1.00    350568



In [ ]:

# NÃO MEXER NESSA CÉLULA
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)


In [29]:
shap_values[1].shape

(80, 3)

In [ ]:
shap.force_plot(explainer.expected_value[0], shap_values[1][0,:], X_train.iloc[0,:],matplotlib=True)

DimensionError: Length of features is not equal to the length of shap_values!